# 03 — v3 best recurrent system

Objetivo: buscar el mejor sistema posible **sin salir del universo recurrente** y sin comprometer el protocolo leak-free.

Diseño v3:
- dos ramas recurrentes
  - HRV
  - morfología manual por latido
- búsqueda dirigida de hiperparámetros
- comparación corta de familias GRU/LSTM
- selección por **macro-F1 en validación**

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

import itertools
import numpy as np
import pandas as pd
import tensorflow as tf

from src.utils.reproducibility import set_global_seed
from src.features.sequence_builders import SequenceBuildConfig, build_dual_branch_dataset
from src.data.splits import make_fixed_grouped_splits
from src.features.preprocessing import fit_sequence_scaler
from src.training.train import TrainingConfig, fit_model
from src.training.search import run_directed_search
from src.evaluation.metrics import build_evaluation_bundle
from src.evaluation.reporting import plot_confusion, plot_training_history, measure_inference_time
from src.xai.gradients import saliency_map, integrated_gradients
from src.xai.occlusion import temporal_occlusion_importance
from src.utils.io import save_json, save_joblib

In [ ]:
set_global_seed(42)

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "mit-bih-arrhythmia-database-1.0.0"
OUTPUT_DIR = PROJECT_ROOT / "models" / "v3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CFG = SequenceBuildConfig(
    data_dir=str(DATA_DIR),
    label_mode="aami_5",
    n_steps=20,
    horizon=1,
    lead=0,
    beat_window_before=90,
    beat_window_after=162,
    exclude_paced_records=False,
)

In [ ]:
dataset = build_dual_branch_dataset(DATA_CFG)
X_hrv = dataset["X_hrv"]
X_morph = dataset["X_morph"]
y = dataset["y"]
groups = dataset["groups"]
class_names = {int(k): v for k, v in dataset["class_names"].items()}

split = make_fixed_grouped_splits(y=y, groups=groups, seed=42)

X_hrv_train_raw = X_hrv[split.train_idx]
X_hrv_val_raw = X_hrv[split.val_idx]
X_hrv_test_raw = X_hrv[split.test_idx]

X_morph_train_raw = X_morph[split.train_idx]
X_morph_val_raw = X_morph[split.val_idx]
X_morph_test_raw = X_morph[split.test_idx]

y_train = y[split.train_idx]
y_val = y[split.val_idx]
y_test = y[split.test_idx]

In [ ]:
scaler_hrv = fit_sequence_scaler(X_hrv_train_raw, scaler_name="robust")
scaler_morph = fit_sequence_scaler(X_morph_train_raw, scaler_name="robust")

X_hrv_train = scaler_hrv.transform(X_hrv_train_raw)
X_hrv_val = scaler_hrv.transform(X_hrv_val_raw)
X_hrv_test = scaler_hrv.transform(X_hrv_test_raw)

X_morph_train = scaler_morph.transform(X_morph_train_raw)
X_morph_val = scaler_morph.transform(X_morph_val_raw)
X_morph_test = scaler_morph.transform(X_morph_test_raw)

save_joblib(scaler_hrv, OUTPUT_DIR / "scaler_hrv.joblib")
save_joblib(scaler_morph, OUTPUT_DIR / "scaler_morph.joblib")

## Espacio de búsqueda reducido pero útil

No buscamos el modelo más grande. Buscamos el mejor compromiso entre generalización, coste y estabilidad.

In [ ]:
candidate_cfgs = []
for rnn_type, balance_strategy, units_hrv, units_morph, use_attention in [
    ("gru", "class_weights", (64, 32), (64, 32), True),
    ("gru", "focal", (64, 32), (64, 32), True),
    ("gru", "balanced_batch", (64, 32), (64, 32), True),
    ("lstm", "class_weights", (64, 32), (64, 32), True),
    ("gru", "class_weights", (96, 48), (64, 32), True),
    ("gru", "class_weights", (64, 32), (64, 32), False),
]:
    candidate_cfgs.append(
        TrainingConfig(
            model_version="v3",
            batch_size=128,
            epochs=60,
            learning_rate=1e-3,
            clipnorm=1.0,
            balance_strategy=balance_strategy,
            patience=10,
            lr_patience=4,
            checkpoint_path=None,
            random_state=42,
            model_kwargs={
                "rnn_type": rnn_type,
                "units_hrv": units_hrv,
                "units_morph": units_morph,
                "dropout_rate": 0.3,
                "recurrent_dropout_rate": 0.15,
                "use_attention": use_attention,
                "l2_reg": 1e-4,
            },
        )
    )

len(candidate_cfgs)

In [ ]:
search_df, best_model, best_cfg = run_directed_search(
    candidate_cfgs=candidate_cfgs,
    x_train={"hrv_input": X_hrv_train, "morph_input": X_morph_train},
    y_train=y_train,
    x_val={"hrv_input": X_hrv_val, "morph_input": X_morph_val},
    y_val=y_val,
    input_shapes={
        "hrv_input": X_hrv_train.shape[1:],
        "morph_input": X_morph_train.shape[1:],
    },
    num_classes=len(class_names),
)

search_df

## Ablation corta

Esta tabla resume lo que sí conviene comparar en un TFG:
- GRU vs LSTM
- atención sí/no
- estrategia de desbalance
- tamaño moderado de ramas

In [ ]:
search_df[[
    "search_iteration",
    "balance_strategy",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "macro_precision",
    "macro_recall",
    "model_kwargs",
]]

In [ ]:
y_test_prob = best_model.predict({"hrv_input": X_hrv_test, "morph_input": X_morph_test}, verbose=0)
y_test_pred = np.argmax(y_test_prob, axis=1)

eval_bundle = build_evaluation_bundle(
    y_true=y_test,
    y_pred=y_test_pred,
    y_prob=y_test_prob,
    class_names=class_names,
)
eval_bundle["search_results"] = search_df.to_dict(orient="records")
eval_bundle["selected_config"] = {
    "balance_strategy": best_cfg.balance_strategy,
    "model_kwargs": best_cfg.model_kwargs,
}
eval_bundle["inference_time"] = measure_inference_time(
    best_model,
    {"hrv_input": X_hrv_test[:1], "morph_input": X_morph_test[:1]},
    repeats=20,
)
eval_bundle["global_metrics"]

In [ ]:
plot_confusion(y_test, y_test_pred, class_names, normalize=False)

In [ ]:
plot_confusion(y_test, y_test_pred, class_names, normalize=True)

## XAI en sistema de dos ramas

En v3 inspeccionamos por separado:
- atribuciones HRV
- atribuciones morfológicas
- relevancia temporal por oclusión en cada rama

In [ ]:
sample_idx = 0
sample_inputs = {
    "hrv_input": X_hrv_test[sample_idx:sample_idx+1],
    "morph_input": X_morph_test[sample_idx:sample_idx+1],
}

sal = saliency_map(best_model, sample_inputs)
ig = integrated_gradients(best_model, sample_inputs, steps=32)
occ = temporal_occlusion_importance(best_model, sample_inputs, window=1)

print({k: np.asarray(v).shape for k, v in sal.items()})
print({k: np.asarray(v).shape for k, v in ig.items()})
print({k: np.asarray(v).shape for k, v in occ.items()})

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 8))

axes[0, 0].imshow(np.abs(sal["hrv_input"][0]).T, aspect="auto", origin="lower")
axes[0, 0].set_title("Saliency HRV")

axes[0, 1].imshow(np.abs(ig["hrv_input"][0]).T, aspect="auto", origin="lower")
axes[0, 1].set_title("IG HRV")

axes[0, 2].bar(np.arange(len(occ["hrv_input"])), occ["hrv_input"])
axes[0, 2].set_title("Occlusion HRV")

axes[1, 0].imshow(np.abs(sal["morph_input"][0]).T, aspect="auto", origin="lower")
axes[1, 0].set_title("Saliency Morph")

axes[1, 1].imshow(np.abs(ig["morph_input"][0]).T, aspect="auto", origin="lower")
axes[1, 1].set_title("IG Morph")

axes[1, 2].bar(np.arange(len(occ["morph_input"])), occ["morph_input"])
axes[1, 2].set_title("Occlusion Morph")

plt.tight_layout()
plt.show()

In [ ]:
metadata = {
    "model_version": "v3",
    "task": "next_beat_aami5_grouped",
    "class_names": class_names,
    "input_shapes": {
        "hrv_input": list(X_hrv_train.shape[1:]),
        "morph_input": list(X_morph_train.shape[1:]),
    },
    "feature_names_hrv": dataset["feature_names_hrv"],
    "feature_names_morph": dataset["feature_names_morph"],
    "split_protocol": "grouped_stratified_record_split",
    "selected_balance_strategy": best_cfg.balance_strategy,
    "loss_name": best_cfg.loss_name,
    "seed": best_cfg.random_state,
    "data_config": dataset["config"],
}

best_model.save(OUTPUT_DIR / "model.keras")
save_json(metadata, OUTPUT_DIR / "metadata.json")
save_json(eval_bundle, OUTPUT_DIR / "metrics_test.json")

## Cierre v3

La configuración elegida debe justificarse por:
- macro-F1
- consistencia por registro
- coste de inferencia razonable
- interpretabilidad más rica